# Hankel Transform of a Sphere Form Factor

This notebook demonstrates how to compute the Hankel transform of the built-in `sphere` form factor using LibHankel.

The sphere form factor is:

$$F(q) = 3 \frac{\sin(qR) - qR\cos(qR)}{(qR)^3}$$

where $R$ is the sphere radius and $q$ is the scattering vector. The Hankel transform ($\nu = 0$) of $F(q)$ gives the correlation function $G(r)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import libhankel

## Set up parameters

The `sphere` form factor takes two parameters: the radius $R$ and a scale factor $\Delta\rho$ (contrast).

In [ ]:
nu = 0                            # Hankel order
params = np.array([10.0, 1.0])    # [radius R, contrast delta_rho]

r_arr = np.linspace(1.0, 25.0, 50)

## Compute the transform with FFTLog

FFTLog is the recommended default strategy. It needs no convergence parameters — pass an empty dict.

In [ ]:
result_fftlog = libhankel.hankel_transform(
    nu, "sphere", r_arr, params, "FFTLog", {}
)

print("G(r) via FFTLog:", result_fftlog)

## Compare with an adaptive strategy

The adaptive `Adaptive_DE_Ooura` strategy uses per-point quadrature. It accepts `n_eval` and `eps_rel` to control accuracy.

In [ ]:
strategy_params = {"n_eval": 250, "eps_rel": 1e-9}

result_adaptive = libhankel.hankel_transform(
    nu, "sphere", r_arr, params, "Adaptive_DE_Ooura", strategy_params
)

print("G(r) via Adaptive_DE_Ooura:", result_adaptive)

## Plot results

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

fftlog = np.array(result_fftlog)
adaptive = np.array(result_adaptive)

ax.plot(r_arr, fftlog, label="FFTLog")
ax.plot(r_arr, adaptive, linestyle="--", label="Adaptive_DE_Ooura")
ax.set_xlabel("r")
ax.set_ylabel("G(r)")
ax.set_title("Hankel transform of sphere form factor")
ax.legend()

plt.tight_layout()
plt.show()

## Runtime comparison

We time both strategies over several repeated runs to get a stable estimate.

In [ ]:
import timeit

n_repeats = 10

t_fftlog = timeit.timeit(
    lambda: libhankel.hankel_transform(nu, "sphere", r_arr, params, "FFTLog", {}),
    number=n_repeats,
) / n_repeats

t_adaptive = timeit.timeit(
    lambda: libhankel.hankel_transform(nu, "sphere", r_arr, params, "Adaptive_DE_Ooura", strategy_params),
    number=n_repeats,
) / n_repeats

print(f"FFTLog:           {t_fftlog*1e3:.2f} ms")
print(f"Adaptive_DE_Ooura: {t_adaptive*1e3:.2f} ms")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

strategies = ["FFTLog", "Adaptive_DE_Ooura"]
times_ms = [t_fftlog * 1e3, t_adaptive * 1e3]

bars = ax.bar(strategies, times_ms)
ax.bar_label(bars, fmt="{:.2f} ms", padding=4)
ax.set_ylim(0, max(times_ms) * 1.3)
ax.set_ylabel("Mean runtime (ms)")
ax.set_title(f"Runtime comparison (n={len(r_arr)} points, averaged over {n_repeats} runs)")

plt.tight_layout()
plt.show()

## Accuracy comparison

LibHankel provides a closed-form analytical solution for the sphere SESANS correlation function:

$$G(z) = \pi \eta^2 R^4 \left[ \sqrt{1-\xi^2}(2+\xi^2) + \xi^2(4-\xi^2) \ln\frac{\xi}{1+\sqrt{1-\xi^2}} - 2 \right], \quad \xi = \frac{z}{2R}$$

with $G(z)$ flat for $z \ge 2R$. The analytical formula gives $G(z) - G(0)$, and includes a $1/(2\pi)$ normalisation that `hankel_transform` does not apply. We therefore compute $[G(r) - G(0)] / 2\pi$ from the numerical results before comparing, where $G(0)$ is evaluated via `hankel_G0`.

In [ ]:
def analytical_sphere(r_arr, R, eta):
    xi = r_arr / (2.0 * R)
    eps = 1e-15
    pref = eta**2 * np.pi * R**4
    G = np.zeros_like(r_arr, dtype=float)
    last_val = 0.0
    have_last = False
    for i, x in enumerate(xi):
        if 0.0 <= x < 1.0:
            xi2 = x**2
            sqrt_term = np.sqrt(1.0 - xi2)
            term1 = sqrt_term * (2.0 + xi2)
            log_arg = max(x / (1.0 + sqrt_term), eps)
            term2 = xi2 * (4.0 - xi2) * np.log(log_arg)
            G[i] = pref * (term1 + term2 - 2.0)
            last_val = G[i]
            have_last = True
    if have_last:
        for i, x in enumerate(xi):
            if not (0.0 <= x < 1.0):
                G[i] = last_val
    return G

R, eta = params[0], params[1]
G_analytical = analytical_sphere(r_arr, R, eta)

# Subtract G(0) and normalise by 2π to match the analytical convention
G0 = libhankel.hankel_G0("sphere", params, 1e-9)
fftlog_shifted   = (fftlog   - G0) / (2 * np.pi)
adaptive_shifted = (adaptive - G0) / (2 * np.pi)

err_fftlog   = np.max(np.abs(fftlog_shifted   - G_analytical))
err_adaptive = np.max(np.abs(adaptive_shifted - G_analytical))

scale = np.max(np.abs(G_analytical))
rel_fftlog   = err_fftlog   / scale
rel_adaptive = err_adaptive / scale

print(f"{'Strategy':<22} {'Max abs error':>16} {'Max rel error':>16}")
print("-" * 56)
print(f"{'FFTLog':<22} {err_fftlog:>16.3e} {rel_fftlog:>16.3e}")
print(f"{'Adaptive_DE_Ooura':<22} {err_adaptive:>16.3e} {rel_adaptive:>16.3e}")